In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import torch
import pandas as pd
import datetime

DATA_PATH = "E:/ML/UBC"

In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [3]:
with open(os.path.join(DATA_PATH,"./FeatureDataEdgeNext.pkl"), "rb") as f:
    zippedData = pickle.load(f)
    X,y = zip(*zippedData)

In [4]:
X = np.array(X)
# X = np.max(X,1)
y = np.array(y)
print(X.shape)
print(y.shape)

(538, 49, 584)
(538,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)
y_train.shape

(376,)

In [6]:
from sklearn.preprocessing import LabelEncoder

trainList = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
trainList.head()

uniqueLabels = trainList["label"].unique()

enc = LabelEncoder()
enc.fit(np.append(uniqueLabels, "Other"))
# enc.fit(uniqueLabels)
print(enc.classes_)
print(enc.transform(["LGSC"]))


['CC' 'EC' 'HGSC' 'LGSC' 'MC' 'Other']
[3]


## LSTM Model

In [7]:
class LSTMModel(torch.nn.Module):
    def __init__(self, inputSize, outputClasses):
        super().__init__()
        self.name=f"LSTMModel"
        self.gru = torch.nn.GRU(input_size=inputSize, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True)
        self.linear1 = torch.nn.Linear(256, 32)
        self.act = torch.nn.SiLU()
        self.bn = torch.nn.BatchNorm1d(32)
        self.linear2 = torch.nn.Linear(32, outputClasses)
    def forward(self, x):
        x, _ = self.gru(x)
        x = x[:, -1, :]
        x = self.linear1(x)
        x = self.act(x)
        x = self.bn(x)
        x = self.linear2(x)
        return x

In [8]:
model = LSTMModel(X.shape[-1], len(enc.classes_))

model = model.to(device)
model(torch.randn((8,49,584)).to(device)).shape

torch.Size([8, 6])

In [9]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE=32

tensor_x_train = torch.Tensor(X_train) # transform to torch tensor
tensor_y_train = torch.Tensor(y_train).type(torch.int64)
trainDataset = TensorDataset(tensor_x_train,tensor_y_train) # create your datset
trainLoader = DataLoader(trainDataset, batch_size=BATCH_SIZE, shuffle=False)

tensor_x_val = torch.Tensor(X_val) # transform to torch tensor
tensor_y_val = torch.Tensor(y_val).type(torch.int64)
valDataset = TensorDataset(tensor_x_val,tensor_y_val) # create your datset
valLoader = DataLoader(valDataset, batch_size=BATCH_SIZE, shuffle=False)

In [10]:
for XData, targets in trainLoader:
# for X, targets, targetsE,targetsK,targetsL,targetsS, patIds in trainLoader:
    print(f"Shape of X: {XData.shape} {XData.dtype}")
    print(f"Shape of target: {targets.shape} {targets.dtype}")
    break

Shape of X: torch.Size([32, 49, 584]) torch.float32
Shape of target: torch.Size([32]) torch.int64


In [14]:
from torcheval import metrics
from torch.utils.tensorboard import SummaryWriter
from torchmetrics import AUROC, Accuracy


LOG_INTERVAL=2
epochs = 100
startEpoch=0
EARLY_STOPPING_PATIENCE=20
saveModel=False


log_dir = "./logs/"+model.name+"/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

summary_writer = SummaryWriter(log_dir)

# Instantiate an optimizer .
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# Instantiate a loss function.
lossFn = torch.nn.CrossEntropyLoss()
# lossFn = FocalLoss()


accMetric = Accuracy(task="multiclass", num_classes=len(enc.classes_), average="micro").to(device)
aurocMetric = AUROC(task="multiclass", num_classes=len(enc.classes_)).to(device)

accMetricVal = Accuracy(task="multiclass", num_classes=len(enc.classes_), average=None).to(device)


def train(dataloaderTrain, model, optimizer, epoch):
    size = len(dataloaderTrain.dataset)
    model.train()
    for batch, (XTrain, yTrain) in enumerate(dataloaderTrain):
        # XTrain = XTrain.movedim(-1,1)
        XTrain, yTrain = XTrain.to(device), yTrain.to(device)
        pred = model(XTrain)
        # targetLabel = torch.argmax(y, dim=1)
        
        loss = lossFn(pred, yTrain)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # scheduler.step()

        accMetric.update(pred, yTrain)
        aurocMetric.update(pred, yTrain)
        
        if batch % LOG_INTERVAL == 0:
            loss = loss.item()
            print("loss: {:>5f}, acc: {:.4f}, AUROC: {:.4f}  [{:>5d}/{:>5d}]".format(loss, accMetric.compute(),  aurocMetric.compute(), batch*BATCH_SIZE, size))
            summary_writer.add_scalar("Loss", loss, epoch*size//BATCH_SIZE+batch)
            summary_writer.add_scalar("Acc", accMetric.compute(), epoch*size//BATCH_SIZE+batch)
            summary_writer.add_scalar("AUROC", aurocMetric.compute(), epoch*size//BATCH_SIZE+batch)
            summary_writer.flush()



def validate(dataloaderVal, model, epoch):
    print("Start Validation...")
    model.eval()
    loss = 0
    for batch, (XVal, yVal) in enumerate(dataloaderVal):
        # XVal = XVal.movedim(-1,1)
        XVal, yVal = XVal.to(device), yVal.to(device)
        predVal = model(XVal)
        # targetLabel = torch.argmax(y, dim=1)
        lossAdd = lossFn(predVal, yVal)
        loss += lossAdd.detach().cpu().numpy()
        accMetricVal.update(predVal, yVal)
        
    lossVal = loss.item()/(batch+1)
    print("Weighted Avg Cross Entropy: {:>7f}".format(lossVal))
    print("Accuracy: {:>7f}".format(torch.mean(accMetricVal.compute())))
    summary_writer.add_scalar("Val BCE", lossVal, epoch)
    summary_writer.add_scalar("Val Accuracy", torch.mean(accMetricVal.compute()), epoch)
    summary_writer.flush()
    fig, ax = accMetricVal.plot()
    fig.savefig(os.path.join(DATA_PATH, "metricPlots", "accuracy_{}.png".format(epoch)))
    plt.close()
    return torch.mean(accMetricVal.compute())


bestValAcc=0.0
bestEpoch=0

for t in np.arange(startEpoch, startEpoch+epochs):
    print(f"Epoch {t+1}\n-------------------------------")

    train(trainLoader, model, optimizer, t)
    accMetric.reset()
    aurocMetric.reset()

    avgAccVal = validate(valLoader, model, t)
    accMetricVal.reset()

    if saveModel:
        model_scripted = torch.jit.script(model) # Export to TorchScript
        fileName = "{}_epoch{}_CE{:.4f}.pt".format(model.name, t, avgAccVal)
        model_scripted.save(os.path.join(DATA_PATH, fileName))
    
    #Early stopping
    if avgAccVal > bestValAcc:
        bestValAcc = avgAccVal
        bestEpoch = t
        bestWeights = model.state_dict()
    if t - bestEpoch >= EARLY_STOPPING_PATIENCE:
        print("Early stopping")
        break

    print("LR: {:.2E}".format(optimizer.state_dict()["param_groups"][0]["lr"]))
    
print("Done!")
print("Best Val Acc: ", bestValAcc.detach().cpu().numpy())

Epoch 1
-------------------------------
loss: 0.021065, acc: 1.0000, AUROC: 0.8333  [    0/  376]
loss: 0.020481, acc: 1.0000, AUROC: 0.8333  [   64/  376]
loss: 0.017034, acc: 1.0000, AUROC: 0.8333  [  128/  376]
loss: 0.020768, acc: 1.0000, AUROC: 0.8333  [  192/  376]
loss: 0.017968, acc: 1.0000, AUROC: 0.8333  [  256/  376]
loss: 0.025057, acc: 0.9972, AUROC: 0.8333  [  320/  376]
Start Validation...
Weighted Avg Cross Entropy: 0.107458
Accuracy: 0.814286
LR: 1.00E-04
Epoch 2
-------------------------------
loss: 0.020767, acc: 1.0000, AUROC: 0.8333  [    0/  376]


c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028
c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028


loss: 0.018023, acc: 1.0000, AUROC: 0.8333  [   64/  376]
loss: 0.025943, acc: 1.0000, AUROC: 0.8333  [  128/  376]
loss: 0.019070, acc: 1.0000, AUROC: 0.8333  [  192/  376]
loss: 0.017168, acc: 1.0000, AUROC: 0.8333  [  256/  376]
loss: 0.023696, acc: 1.0000, AUROC: 0.8333  [  320/  376]
Start Validation...
Weighted Avg Cross Entropy: 0.091568
Accuracy: 0.818254
LR: 1.00E-04
Epoch 3
-------------------------------
loss: 0.020713, acc: 1.0000, AUROC: 0.8333  [    0/  376]
loss: 0.019296, acc: 0.9896, AUROC: 0.8333  [   64/  376]


c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028
c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028


loss: 0.015459, acc: 0.9937, AUROC: 0.8333  [  128/  376]
loss: 0.018848, acc: 0.9955, AUROC: 0.8333  [  192/  376]
loss: 0.016489, acc: 0.9965, AUROC: 0.8333  [  256/  376]
loss: 0.023150, acc: 0.9972, AUROC: 0.8333  [  320/  376]
Start Validation...
Weighted Avg Cross Entropy: 0.091501
Accuracy: 0.814286
LR: 1.00E-04
Epoch 4
-------------------------------
loss: 0.018757, acc: 1.0000, AUROC: 0.8333  [    0/  376]
loss: 0.017301, acc: 1.0000, AUROC: 0.8333  [   64/  376]
loss: 0.014543, acc: 1.0000, AUROC: 0.8333  [  128/  376]
loss: 0.018251, acc: 1.0000, AUROC: 0.8333  [  192/  376]
loss: 0.015620, acc: 1.0000, AUROC: 0.8333  [  256/  376]
loss: 0.022530, acc: 1.0000, AUROC: 0.8333  [  320/  376]
Start Validation...
Weighted Avg Cross Entropy: 0.074749
Accuracy: 0.818254
LR: 1.00E-04
Epoch 5
-------------------------------
loss: 0.018219, acc: 1.0000, AUROC: 0.8333  [    0/  376]
loss: 0.017209, acc: 1.0000, AUROC: 0.8333  [   64/  376]
loss: 0.014771, acc: 1.0000, AUROC: 0.8333  [ 

...

c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028
c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028


LR: 1.00E-04
Epoch 19
-------------------------------
loss: 0.011280, acc: 1.0000, AUROC: 0.8333  [    0/  376]
loss: 0.010296, acc: 1.0000, AUROC: 0.8333  [   64/  376]
loss: 0.009325, acc: 1.0000, AUROC: 0.8333  [  128/  376]
loss: 0.010764, acc: 1.0000, AUROC: 0.8333  [  192/  376]
loss: 0.009546, acc: 1.0000, AUROC: 0.8333  [  256/  376]
loss: 0.013235, acc: 1.0000, AUROC: 0.8333  [  320/  376]
Start Validation...
Weighted Avg Cross Entropy: 0.075245
Accuracy: 0.814286
LR: 1.00E-04
Epoch 20
-------------------------------


c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028
c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028


loss: 0.010952, acc: 1.0000, AUROC: 0.8333  [    0/  376]
loss: 0.009991, acc: 1.0000, AUROC: 0.8333  [   64/  376]
loss: 0.009062, acc: 1.0000, AUROC: 0.8333  [  128/  376]
loss: 0.010448, acc: 1.0000, AUROC: 0.8333  [  192/  376]
loss: 0.009268, acc: 1.0000, AUROC: 0.8333  [  256/  376]
loss: 0.012845, acc: 1.0000, AUROC: 0.8333  [  320/  376]
Start Validation...
Weighted Avg Cross Entropy: 0.075158
Accuracy: 0.814286
LR: 1.00E-04
Epoch 21
-------------------------------
loss: 0.010635, acc: 1.0000, AUROC: 0.8333  [    0/  376]
loss: 0.009700, acc: 1.0000, AUROC: 0.8333  [   64/  376]
loss: 0.008807, acc: 1.0000, AUROC: 0.8333  [  128/  376]
loss: 0.010147, acc: 1.0000, AUROC: 0.8333  [  192/  376]
loss: 0.009004, acc: 1.0000, AUROC: 0.8333  [  256/  376]
loss: 0.012475, acc: 1.0000, AUROC: 0.8333  [  320/  376]
Start Validation...
Weighted Avg Cross Entropy: 0.075073
Accuracy: 0.814286
LR: 1.00E-04
Epoch 22
-------------------------------
loss: 0.010331, acc: 1.0000, AUROC: 0.8333  

c:\Users\Manuel\anaconda3\envs\torch\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028


In [12]:
model.load_state_dict(bestWeights)

<All keys matched successfully>

In [13]:
torch.save({
    'epoch': t,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    # 'scheduler_state_dict': scheduler.state_dict(),
    # 'loss': loss,
    }, os.path.join("./", model.name+"_epoch_{}.pt".format(bestEpoch)))